# Geographical Analysis

Analyze store geography, adjacency, and location-related behavior.

This notebook is a cleaned and renamed version of the original analysis notebook.

## Notebook Guide
This notebook maps the store network, measures distance between related stores, and checks how geographic proximity relates to shared customer traffic.


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'dataset'
GENERATED_DIR = PROJECT_ROOT / 'generated'
FIGURES_DIR = PROJECT_ROOT / 'figures'

GENERATED_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir: {DATA_DIR}')
print(f'Generated dir: {GENERATED_DIR}')
print(f'Figures dir: {FIGURES_DIR}')


Project root: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis
Dataset dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/dataset
Generated dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated
Figures dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/figures


In [ ]:
# Install the mapping library if the notebook kernel does not already have it.
# Install the mapping library if the notebook kernel does not already have it.
!pip install folium


In [6]:
import pandas as pd
import folium
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, cos, sin, asin, sqrt


ModuleNotFoundError: No module named 'folium'

In [ ]:
df_transactions = pd.read_csv(GENERATED_DIR / 'merged_transactions.csv')
df_transactions.head(6)


In [ ]:
# Build a deduplicated reference table of stores and their coordinates.
unique_stores = df_transactions[[
    'ID_LIEU_DE_VENTE',              
    'LB_DESIGNATION_LIEU_DE_VENTE',
    'LB_VILLE',                      
    'LATITUDE',                    
    'LONGITUDE'                     
]].drop_duplicates().sort_values('ID_LIEU_DE_VENTE')


unique_stores.reset_index(drop=True, inplace=True)

print(f"Number of stores: {len(unique_stores)}")
display(unique_stores)

invalid_coords = unique_stores[
    (unique_stores['LATITUDE'] == 0) | 
    (unique_stores['LONGITUDE'] == 0)
]

if len(invalid_coords) > 0:
    print(f"\nWarning: {len(invalid_coords)} store coordinates look invalid because they are equal to 0.")
    display(invalid_coords)
else:
    print("\nAll store coordinates look valid based on the zero-value check.")


In [ ]:
# Plot each store on an interactive map to validate the geographic coverage.
center_lat = unique_stores['LATITUDE'].mean()
center_lon = unique_stores['LONGITUDE'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

for idx, row in unique_stores.iterrows():
    folium.Marker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        popup=f"{row['LB_DESIGNATION_LIEU_DE_VENTE']} ({row['LB_VILLE']})",
        tooltip=f"Store ID: {row['ID_LIEU_DE_VENTE']}",
        icon=folium.Icon(color='blue', icon='shopping-cart')
    ).add_to(m)

print("Store location map:")
m


In [ ]:
print("Analyzing adjacent stores")

# 1. Use the Haversine formula to estimate distance between two stores.
def calculate_distance(row):
    coord_a = store_coords.get(row['ID_LIEU_DE_VENTE_A'])
    coord_b = store_coords.get(row['ID_LIEU_DE_VENTE_B'])
    
    if not coord_a or not coord_b:
        return None

    lon1, lat1, lon2, lat2 = map(radians, [
        coord_a['LONGITUDE'], coord_a['LATITUDE'], 
        coord_b['LONGITUDE'], coord_b['LATITUDE']
    ])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r


In [ ]:
# 2. Calculate the distance for every connected store pair.

# df_adjacency = links.copy()

print("Calculating distances between store pairs")
df_adjacency['distance_km'] = df_adjacency.apply(calculate_distance, axis=1)

df_adjacency['Store_Name_A'] = df_adjacency['ID_LIEU_DE_VENTE_A'].map(store_names)
df_adjacency['Store_Name_B'] = df_adjacency['ID_LIEU_DE_VENTE_B'].map(store_names)


In [ ]:
# 3. Keep only nearby store pairs that also share customers.
adjacency_threshold_km = 20

adjacent_pairs = df_adjacency[
    (df_adjacency['distance_km'] <= adjacency_threshold_km) & 
    (df_adjacency['shared_customers'] > 0)
].sort_values(by='shared_customers', ascending=False)

print(f"\n--- Top 10 Adjacent Store Pairs (Distance < {adjacency_threshold_km}km) with Highest Shared Traffic ---")
display_cols = ['Store_Name_A', 'Store_Name_B', 'distance_km', 'shared_customers']
display(adjacent_pairs[display_cols].head(10))


In [ ]:
# 4. Measure how distance relates to shared-customer overlap.
plt.figure(figsize=(10, 6))

# Restrict the view to pairs within 100 km so extreme distances do not dominate the trend.
data_filtered = df_adjacency[df_adjacency['distance_km'] < 100]

# Compute the correlation between distance and shared-customer traffic.
corr = data_filtered[['distance_km', 'shared_customers']].corr().iloc[0,1]

sns.regplot(
    data=data_filtered, 
    x='distance_km', 
    y='shared_customers', 
    scatter_kws={'alpha': 0.6}, 
    line_kws={'color': 'red', 'label': f'Linear Trend (Corr: {corr:.2f})'}
)


plt.title(f'Relationship between Distance and Shared Customers (Pairs < 100km)')
plt.xlabel('Distance between Stores (km)')
plt.ylabel('Number of Shared Customers')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend() 
plt.show()

print(f"Correlation between Distance and Shared Customers: {corr:.2f}")


In [ ]:
print(f"Drawing Map for {len(adjacent_pairs)} adjacent store pairs...")

# 1. Prepare the subset of stores involved in the nearby-pair network.
relevant_store_ids = set(adjacent_pairs['ID_LIEU_DE_VENTE_A']).union(set(adjacent_pairs['ID_LIEU_DE_VENTE_B']))

filtered_stores_list = []
for store_id in relevant_store_ids:
    if store_id in store_coords:
        store_data = store_coords[store_id]
        store_data['ID'] = store_id
        store_data['NAME'] = store_names.get(store_id, f"Store {store_id}")
        filtered_stores_list.append(store_data)

# Derive map bounds from the filtered stores, with a fallback extent if needed.
lats = [s['LATITUDE'] for s in filtered_stores_list]
lons = [s['LONGITUDE'] for s in filtered_stores_list]

if lats and lons:
    min_lat, max_lat = min(lats), max(lats)
    min_lon, max_lon = min(lons), max(lons)
else:
    min_lat, max_lat = 42.0, 51.0
    min_lon, max_lon = -5.0, 8.0


In [ ]:
# 2. Initialize the map canvas and fit it to the relevant area.
m = folium.Map(tiles='cartodbpositron') 
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

# 3. Draw one line per nearby pair, weighted by the number of shared customers.
if not adjacent_pairs.empty:
    max_weight = adjacent_pairs['shared_customers'].max()

    for _, row in adjacent_pairs.iterrows():
        loc_a = [store_coords[row['ID_LIEU_DE_VENTE_A']]['LATITUDE'], store_coords[row['ID_LIEU_DE_VENTE_A']]['LONGITUDE']]
        loc_b = [store_coords[row['ID_LIEU_DE_VENTE_B']]['LATITUDE'], store_coords[row['ID_LIEU_DE_VENTE_B']]['LONGITUDE']]
        
        folium.PolyLine(
            locations=[loc_a, loc_b],
            weight=(row['shared_customers'] / max_weight) * 5 + 1, 
            color='#E74C3C',
            opacity=0.7,
            tooltip=f"{row['Store_Name_A']} <-> {row['Store_Name_B']}<br>Shared: {row['shared_customers']}<br>Dist: {row['distance_km']:.1f}km"
        ).add_to(m)

# 4. Add store markers so the network can be read spatially.
for store in filtered_stores_list:
    folium.CircleMarker(
        location=[store['LATITUDE'], store['LONGITUDE']],
        radius=6,
        color='#2980B9', 
        fill=True,
        fill_color='#2980B9',
        popup=f"{store['NAME']} (ID: {store['ID']})"
    ).add_to(m)

# 5. Render the final interactive map.



## Results Summary

- Built a clean store-location reference table and mapped the network of stores on an interactive geographic view.
- Estimated distance between connected store pairs, filtered nearby stores, and examined how proximity relates to shared-customer traffic.
- Produced an interactive adjacency map that highlights nearby stores with overlapping customer bases.
- Main outputs: the store-location map, the nearby-store network map, and the distance-versus-shared-customer diagnostic plot.
